# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/satkynovilgiz/flyrank-ml-content-refresh/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**My lane: Lane 2, Refresh / Content Opportunity Scoring.**

I picked this lane because my ML-01 work pointed straight at it.
Two findings stuck with me.

- **Age and decline are linked, but not the way I expected.** Younger pages were trending down more often than old ones (the numbers are in section 3).
  So "old page = needs a refresh" is too simple.
- **The fixed hand rule barely works.** The cell below recomputes it.
  It flags only 17 of 30,000 pages, and 0 pages on the 7 held-out clients.
  When a rule flags almost nothing, it cannot build a review queue.

A content team has limited editor time.
A better way to order the queue is a real, useful problem, and Lane 2 is exactly that.


In [1]:
# Setup: find the repo root (works from work/notebooks/, the repo root, or Colab) and load the starter CSV.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("flyrank-ml-internship-starter").is_dir():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
root = Path.cwd()
while not (root / "data/raw/content_refresh_anonymized.csv").exists() and root != root.parent:
    root = root.parent
os.chdir(root)

import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].to_numpy()

# ML-01 recap, recomputed here: the hand rule is stale (>= 180 days since update) AND visible (>= 500 impressions).
hand = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).to_numpy()

# Same client holdout as ML-01: GroupShuffleSplit by client_id, 20% of clients held out, seed 42.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups=df["client_id"]))
n_test_clients = df["client_id"].iloc[test_idx].nunique()

print(f"pages: {len(df):,} | clients: {df['client_id'].nunique()}")
print(f"hand rule flags {hand.sum()} of {len(df):,} pages "
      f"({hand.mean():.2%}); declining among flagged: {y[hand].mean():.3f}")
print(f"hand rule flags {hand[test_idx].sum()} of {len(test_idx):,} pages "
      f"on the {n_test_clients} held-out clients")


pages: 30,000 | clients: 32
hand rule flags 17 of 30,000 pages (0.06%); declining among flagged: 0.941
hand rule flags 0 of 6,163 pages on the 7 held-out clients


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**My question:** Which pages should a content team review first, and for which action (refresh, expand, protect, prune, monitor)?

- **The decision:** the order of the review queue. Which pages get an editor's time first.
- **Who acts on it:** a content editor or SEO lead. They open the top of the list, read each page, and choose an action.
- **Why "first" matters:** the cell below shows there are 30,000 pages across 32 clients.
  One client alone has 7,008 pages, and the median client has 567. Nobody reviews all of that.
- **What a wrong call costs:**
  - A false alarm wastes editor hours on a page that would not have benefited.
  - A miss is worse in a quiet way: a declining page sits low in the list and keeps losing traffic.

My work does not make the edit. It decides where a human looks first.


In [2]:
# Scale of the decision: how many pages compete for an editor's attention, per client?
pages_per_client = df.groupby("client_id").size()
print(f"pages per client: min {pages_per_client.min():,} | "
      f"median {pages_per_client.median():,.0f} | max {pages_per_client.max():,}")
print(f"pages trending down: {y.sum():,} of {len(df):,}")


pages per client: min 3 | median 567 | max 7,008
pages trending down: 16,262 of 30,000


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the cell below (aggregates only):

1. **54.2% of all pages are trending down.** That is more than half. "Find the declining ones" alone does not shrink the queue. I need a ranking.
2. **The share trending down changes with age.** It is 60.9% / 64.0% / 59.5% for pages aged 90-119 / 120-179 / 180-269 days.
   It drops to 47.0% for 270-364 days and 42.6% for 365+ days.
   So younger pages decline more often here. This is observed and directional.
   It may be tangled up with which clients have young pages, or with pages settling after launch.
3. **The hand rule cannot rank most pages.** `days_since_last_update >= 180` is true for only 0.58% of pages.
   The rule flags 17 of 30,000 pages, and 0 on held-out clients. Every other page ties at score 0.

Together these say there is a real signal (age, visibility) and a weak baseline to beat.
That is why this lane looks worth the next 7 weeks.


In [3]:
# Quick look: aggregates only (no IDs, no rows).
# Number 1: the share of all pages trending down.
trend = df["trend_direction"].str.lower()
print(f"1) share of pages trending down: {trend.eq('down').mean():.1%}")
print("   full trend mix:", ", ".join(f"{k} {v:.1%}" for k, v in trend.value_counts(normalize=True).items()), "\n")

# Number 2: the share trending down, by content age band (same bands as my ML-01 notebook).
age_bins   = [90, 120, 180, 270, 365, np.inf]
age_labels = ["90-119 d", "120-179 d", "180-269 d", "270-364 d", "365+ d"]
age_band = pd.cut(df["content_age_days"], bins=age_bins, labels=age_labels, right=False)
age_table = pd.DataFrame({
    "pages":      age_band.value_counts().reindex(age_labels),
    "share_down": trend.eq("down").groupby(age_band, observed=False).mean().round(3),
})
age_table.index.name = "age_band"
assert age_table["pages"].sum() == len(df), "every row should land in exactly one band"
print("2) share trending down by age band:")
print(age_table.to_string(), "\n")

# Number 3: how many pages the fixed hand rule can rank at all (everything else ties at 0).
stale_share = (df["days_since_last_update"] >= 180).mean()
print(f"3) days_since_last_update >= 180 is true for {stale_share:.2%} of pages; "
      f"the hand rule flags {hand.sum()} of {len(df):,} pages, "
      f"and {hand[test_idx].sum()} on held-out clients")


1) share of pages trending down: 54.2%
   full trend mix: down 54.2%, stable 19.9%, up 14.6%, new 7.5%, flat 3.8% 

2) share trending down by age band:
           pages  share_down
age_band                    
90-119 d    5459       0.609
120-179 d   6555       0.640
180-269 d   4513       0.595
270-364 d   7113       0.470
365+ d      6360       0.426 

3) days_since_last_update >= 180 is true for 0.58% of pages; the hand rule flags 17 of 30,000 pages, and 0 on held-out clients


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**

- What I **observed** in this anonymized starter slice. Example: in this data, younger pages were trending down more often.
- **Directional** patterns, stated with their size and their limits (one 30,000-row slice, 32 clients, a current-window label).
- A **decision-support ranking**: an ordered list of pages worth a human review first, with a reason for each page.
- **Measured** results on held-out clients, compared against a simple baseline and the base rate.

**What I will never claim:**

- That a refresh **causes** a recovery. This data has no experiment, so it cannot prove cause.
- That I **predict Google** or know how its ranking works.
- That a page is "declining" for sure. My label is a proxy. A drop can be seasonality, consolidation into a sibling page, or noise (guide section 7).
- Anything about a named client, URL or query. I only publish aggregates.


In [4]:
# No code needed for this section: it is about wording, not numbers.
# The claims above stay tied to the aggregates printed in sections 1-3.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.